# 🏀 CoachBot — NBA Player Workout Generator
**RAG Pipeline using LangChain + Google Gemini**

Search any NBA player → CoachBot finds their drills, quotes, and training style → Generates a full 60-min workout for your athlete.

---
### Architecture (Controls Engineer View)
```
Query (Player Name)
    → Embeddings (A/D Conversion: text → numbers)
    → Chroma Vector DB (Tag Database: find closest match)
    → Retrieved Context (Recipe lookup)
    → Gemini LLM (HMI Logic Layer: generate workout)
    → Structured 60-min Workout Output
```

## Cell 1 — Install Dependencies
Run this once. These are your 'libraries' — like importing function blocks in TIA Portal.

In [ ]:
# Run this cell once to install all required packages
# Think of this like installing drivers for your PLC software

%pip install langchain langchain-google-genai langchain-chroma langchain-community
%pip install chromadb google-generativeai youtube-search-python requests beautifulsoup4

## Cell 2 — Configuration & API Key
Set your Google API key here. Get one free at https://aistudio.google.com

In [ ]:
import os

# ─────────────────────────────────────────────
# 🔑 PASTE YOUR GOOGLE API KEY HERE
# Get one at: https://aistudio.google.com → Get API Key
# ─────────────────────────────────────────────
os.environ["GOOGLE_API_KEY"] = "AIza-YOUR-KEY-HERE"

print("✅ API key set.")

## Cell 3 — NBA Player Knowledge Base
This is your **document library** — like a PLC tag database or recipe table.

Each player has:
- 🏋️ Signature drills they are known for
- 💬 Real training quotes
- 🎥 YouTube search terms to find their workout videos
- 🧠 Key skills / style tags

You can add more players anytime by following the same format.

In [ ]:
from langchain_core.documents import Document

# ─────────────────────────────────────────────────────────────────
# 📚 KNOWLEDGE BASE — Add players here in the same format
# Each entry becomes a 'document' that gets embedded into the DB
# ─────────────────────────────────────────────────────────────────

player_data = [

    # ─── STEPH CURRY ───────────────────────────────────────────
    Document(
        page_content="""
        Player: Steph Curry | Team: Golden State Warriors | Position: Point Guard

        SIGNATURE DRILLS:
        - Mikan Drill (both hands, at game speed): Builds finishing at the rim with either hand.
        - Ball Handling Circuit: Two-ball dribbling, figure-8s, behind-the-back, crossover combo. 5 minutes continuous.
        - Catch-and-Shoot off screens: Coach or partner sets a screen, player curls, catches, and fires. 50 makes.
        - 3-point shooting from 5 spots: Corners, wings, top of the key. Must make 10 from each spot before moving.
        - Off-dribble pull-up: Dribble hard left, stop on a dime, pull-up mid-range. Repeat right. 40 makes total.
        - Steph's 'runway' shooting drill: Start at half court, dribble full speed, pull up for 3 at the arc. Simulates his step-back.

        TRAINING QUOTES:
        - 'Success is not an accident. Success is actually a choice.'
        - 'I've worked too hard and too long to let anything stand in the way of my goals.'
        - 'Shooting is about muscle memory. You have to do it 1,000 times to do it once in a game.'
        - 'I try to be efficient in everything I do — every rep, every move has a purpose.'

        KEY SKILLS: Ball handling, off-screen shooting, quick release, footwork, shooting off the dribble, conditioning.

        YOUTUBE SEARCH TERMS:
        - 'Steph Curry full workout training'
        - 'Steph Curry shooting drill routine NBA'
        - 'Steph Curry ball handling workout'
        """,
        metadata={"player": "Steph Curry", "position": "PG", "skills": "shooting, ball handling"}
    ),

    # ─── KOBE BRYANT ───────────────────────────────────────────
    Document(
        page_content="""
        Player: Kobe Bryant | Team: LA Lakers | Position: Shooting Guard

        SIGNATURE DRILLS:
        - Post footwork series: Drop step, up-and-under, turnaround fadeaway. 20 makes from each side.
        - The Kobe 'triangle' shooting drill: Mid-range shots from elbow, baseline, and top. Must make 50 mid-range before 3s.
        - Mamba mentality conditioning: 5 minutes of full-court sprints at game pace with ball in hand.
        - 1-on-1 iso footwork: Jab step series — jab and shoot, jab and drive, jab and crossover. 20 reps each move.
        - Film study + replication: Watch a scoring sequence on film, then replicate every move on the court.
        - Early morning 4AM shooting: 800 shots before anyone else arrives. Pure volume and consistency.

        TRAINING QUOTES:
        - 'The mindset isn't about seeking a result — it's about the process of getting to that result.'
        - 'Everything negative — pressure, challenges — is all an opportunity for me to rise.'
        - 'If you're afraid to fail, then you're probably going to fail.'
        - '4AM. Every day. That's when the work starts.'

        KEY SKILLS: Post play, mid-range shooting, footwork, iso scoring, mental toughness, fadeaway.

        YOUTUBE SEARCH TERMS:
        - 'Kobe Bryant workout training routine'
        - 'Kobe Bryant footwork drills post moves'
        - 'Kobe Bryant Mamba mentality practice'
        """,
        metadata={"player": "Kobe Bryant", "position": "SG", "skills": "post play, mid-range, footwork"}
    ),

    # ─── KEVIN DURANT ──────────────────────────────────────────
    Document(
        page_content="""
        Player: Kevin Durant | Team: Phoenix Suns | Position: Small Forward

        SIGNATURE DRILLS:
        - Shoot over contact: Coach holds a foam pad above shooter's release point. Forces high arc.
        - KD's mid-range series: Step-back from the elbow, lean-in from baseline, turnaround from the block. 30 makes each.
        - Ball handling for bigs: Full dribbling circuits normally run by guards — forces tall players to handle under control.
        - Catch-and-shoot off movement: Sprint, come off pin-down, catch, shoot. 60 makes total.
        - Post-to-perimeter combo: Start in the post, face up, pump fake, drive — or pop to the 3. Decision-making drill.
        - Free throw focus: 100 free throws, tracking make percentage. Nothing leaves the gym under 85%.

        TRAINING QUOTES:
        - 'Hard work beats talent when talent fails to work hard.'
        - 'I just want to get better every single day. That's all I think about.'
        - 'The most important thing is to stay patient and trust your process.'
        - 'Shooting is my gift. But I put in work every single day to keep it sharp.'

        KEY SKILLS: Shooting over defenders, mid-range mastery, scoring in traffic, versatility, free throws.

        YOUTUBE SEARCH TERMS:
        - 'Kevin Durant workout training drills'
        - 'KD shooting drill mid-range routine'
        - 'Kevin Durant skill development session'
        """,
        metadata={"player": "Kevin Durant", "position": "SF", "skills": "scoring, shooting, mid-range"}
    ),

    # ─── LEBRON JAMES ──────────────────────────────────────────
    Document(
        page_content="""
        Player: LeBron James | Team: LA Lakers | Position: Small Forward / PG

        SIGNATURE DRILLS:
        - Full-court transition finishing: Start at baseline, push pace full court, finish with either hand at rim. 20 reps.
        - Playmaking vision drill: 3-on-2 fast break reads — coach calls out scenarios mid-play, player must react.
        - LeBron body conditioning: Mix of plyometrics, resistance band work, and court sprints. 30-minute block.
        - Post-up and kick-out: Back down defender in post, read double team, kick to shooter. Decision making under pressure.
        - Ball handling to layup: Crossover, behind-back, hesitation combo ending in a strong layup or euro step.
        - Recovery and flexibility: 30-minute stretching and recovery routine. LeBron spends $1M/year on his body.

        TRAINING QUOTES:
        - 'I treat every practice like it's a championship game.'
        - 'You have to be able to accept failure to get better.'
        - 'My body is my business. I invest in it every single day.'
        - 'The only way to get better is to surround yourself with people who make you work harder.'

        KEY SKILLS: Athleticism, playmaking, finishing at rim, transition, court vision, conditioning.

        YOUTUBE SEARCH TERMS:
        - 'LeBron James workout training routine'
        - 'LeBron James full court drills athleticism'
        - 'LeBron James skill training session'
        """,
        metadata={"player": "LeBron James", "position": "SF/PG", "skills": "athleticism, playmaking, finishing"}
    ),

    # ─── NIKOLA JOKIC ──────────────────────────────────────────
    Document(
        page_content="""
        Player: Nikola Jokic | Team: Denver Nuggets | Position: Center

        SIGNATURE DRILLS:
        - Passing from the post: From high post and low post, make 10 pinpoint passes to cutters. Then reverse.
        - Big man ball handling: Full guard-level dribbling circuits — forces bigs to build hand coordination.
        - Shooting off movement at center range: Floaters, short mid-range, face-up jumpers to 15 feet. 40 makes.
        - Pick-and-roll reads: Jokic sets the screen, rolls, reads whether to catch-and-finish or kick back out.
        - IQ simulation: Watch film of a possession, pause it, explain where every player should move. Build basketball IQ.
        - Conditioning: Jokic is deceptively fit — side shuffles, drop steps under fatigue. Work at game speed not just rest speed.

        TRAINING QUOTES:
        - 'I just want to win. The stats will come if you focus on winning.'
        - 'Basketball is simple. See the open man. Make the right play.'
        - 'The highest IQ play is usually the right play.'

        KEY SKILLS: Passing, IQ, post play, pick-and-roll, floater, vision, unselfishness.

        YOUTUBE SEARCH TERMS:
        - 'Nikola Jokic skill training drills'
        - 'Jokic passing drill post play workout'
        - 'Nikola Jokic full workout session'
        """,
        metadata={"player": "Nikola Jokic", "position": "C", "skills": "passing, IQ, post play"}
    ),

    # ─── JALEN BRUNSON ─────────────────────────────────────
    Document(
        page_content="""
        Player: Jalen Brunson | Team: New York Knicks | Position: Point Guard

        SIGNATURE DRILLS:
        - Snatch-back to pull-up: Drive hard at the defender, snatch the ball back to create space, rise into a midrange jumper. 30 makes each side.
        - Catch-and-shoot reps: Off screens, off relocations, and in transition. Hundreds of reps per session to sharpen timing and release speed.
        - Pivot and footwork series: Pivot out of the post, up-fake into a jab step, counter off the catch. Builds the ability to operate in tight spaces.
        - The 'broom drill': A partner holds a long pole/broom overhead to contest the shot, forcing a higher, quicker release arc.
        - Touch shot reps: Floaters and short pull-ups around the paint, focused on soft touch over bigger defenders.
        - 1-on-1 to a make: Live one-on-one work where the defender plays full speed, building decision-making under real pressure.

        TRAINING QUOTES:
        - 'Our training sessions are very fundamental.' (Brunson's trainer, Dave Williams)
        - 'We definitely work on pivots, footwork, and touch shots. We get into all of that.' (Dave Williams)

        KEY SKILLS: Footwork, pivoting, midrange shooting, catch-and-shoot mechanics, scoring in tight spaces, decision-making.

        YOUTUBE SEARCH TERMS:
        - 'Jalen Brunson workout training drills'
        - 'Jalen Brunson footwork pivot drills'
        - 'Jalen Brunson signature moves snatch back'
        """,
        metadata={"player": "Jalen Brunson", "position": "PG", "skills": "footwork, midrange, pivoting"}
    ),

    # ─── SHAI GILGEOUS-ALEXANDER ─────────────────────────────────────
    Document(
        page_content="""
        Player: Shai Gilgeous-Alexander | Team: Oklahoma City Thunder | Position: Point Guard

        SIGNATURE DRILLS:
        - Deceleration drill: Sprint at full speed, then stop on a dime under control. Builds the strength to absorb high-impact stops without losing balance.
        - Balance and stability work: Single-leg holds, pivots, and contorting movements while staying grounded — trains body control for drawing fouls and changing pace.
        - Two-ball dribbling under pressure: Combine two-ball dribbling with rapid direction changes to simulate defensive pressure in confined spaces.
        - Agility ladder footwork: Fast-feet ladder patterns to build quickness for elite guard movement.
        - Change-of-pace drive: Practice driving at one speed, then exploding to a completely different speed mid-drive to throw off defenders' timing.
        - Strength foundation: Bulgarian split squats and RDLs to build the strength needed to absorb contact at full speed.

        TRAINING QUOTES:
        - 'He already had a routine. He was already working... He had a paper that he wrote drills on.' (on SGA's youth training habits, via his former coach)

        KEY SKILLS: Change of pace, balance, body control, deceleration, footwork, drawing contact.

        YOUTUBE SEARCH TERMS:
        - 'Shai Gilgeous-Alexander workout training drills'
        - 'SGA footwork balance training'
        - 'Shai Gilgeous-Alexander signature moves'
        """,
        metadata={"player": "Shai Gilgeous-Alexander", "position": "PG", "skills": "change of pace, balance, footwork"}
    ),

    # ─── JAYSON TATUM ─────────────────────────────────────
    Document(
        page_content="""
        Player: Jayson Tatum | Team: Boston Celtics | Position: Forward

        SIGNATURE DRILLS:
        - Floater off the catch: Catch, take 1-2 dribbles, pick the ball up low, and rise for a floater over a defender. Repeat from both sides.
        - In-and-out side step: Practice the in-and-out dribble move into a side-step jumper, used to create separation from a closing defender.
        - Step-back into 'punch drag': Combine a hard dribble with a step-back to create space for a clean jumper.
        - Post-up footwork: Get solid post positioning, work counters from the elbow, using size advantage with a high-release jumpshot.
        - Deadlift-to-court transition: Heavy lower body strength work immediately followed by on-court dribbling, shooting, and running drills to train power under fatigue.

        TRAINING QUOTES:
        - On his approach to recovery and training: relentless, methodical work translated his rehab into renewed explosiveness on the floor.

        KEY SKILLS: Scoring versatility, floaters, step-back shooting, post footwork, size and strength utilization.

        YOUTUBE SEARCH TERMS:
        - 'Jayson Tatum workout training drills'
        - 'Jayson Tatum signature moves step back'
        - 'Jayson Tatum floater post move drill'
        """,
        metadata={"player": "Jayson Tatum", "position": "F", "skills": "scoring versatility, floaters, step-back"}
    ),

    # ─── KLAY THOMPSON ─────────────────────────────────────
    Document(
        page_content="""
        Player: Klay Thompson | Team: Golden State Warriors / Dallas Mavericks | Position: Shooting Guard

        SIGNATURE DRILLS:
        - Zero-dribble catch-and-shoot: Catch the pass and shoot immediately with no dribble, repeated from 5 spots around the arc. Builds quick-release mechanics.
        - Quick catch into a single move: Catch on the move, make one quick decision (shoot, one-dribble pull-up, or hesitation), and finish. No wasted motion.
        - Transition three reps: Run the floor in transition and spot up immediately behind the arc instead of attacking the rim — trains floor spacing and conditioning together.
        - Pump fake jab into one-dribble shot: Catch at the three-point line, pump fake, jab to move the defender, then rise into a one-dribble jumper.
        - Footwork off the catch: Plant the inside foot before catching while moving left; anchor and step into the shot while stationary. Repeat both directions.
        - Conditioning + free throws: Burpees paired immediately with free throws to practice shooting under fatigue.

        TRAINING QUOTES:
        - 'I don't adjust my routine to the opponent. I try to make the defense adjust to me, rather than adjust to them.'
        - 'I used to shoot a lot more before the game... I cut my routine in half, and my shooting percentage went up.'

        KEY SKILLS: Catch-and-shoot, off-ball movement, quick release, shooting footwork, conditioning while shooting.

        YOUTUBE SEARCH TERMS:
        - 'Klay Thompson catch and shoot drills'
        - 'Klay Thompson shooting form workout'
        - 'Klay Thompson transition shooting drill'
        """,
        metadata={"player": "Klay Thompson", "position": "SG", "skills": "catch-and-shoot, quick release, off-ball movement"}
    ),

    # ─── KYRIE IRVING ─────────────────────────────────────
    Document(
        page_content="""
        Player: Kyrie Irving | Team: Dallas Mavericks | Position: Point Guard

        SIGNATURE DRILLS:
        - Two-ball partner awareness drill: Dribble one ball while a partner randomly tosses a second ball at you from 10 feet away. Catch it one-handed, pass back, and keep your original dribble alive. Builds split focus and ball control under chaos.
        - The 'Kyrie Irving drill': Start at half court, dribble at speed to the 3-point line while passing back and forth with a coach using your weak hand 3 times, then finish at the rim or with a short pull-up jumper.
        - Ambidextrous handle work: Off-hand dribbling drills specifically to make the off hand as active and protective as the dominant hand.
        - Finishing with either hand: Practice scripted finishing sequences within 8 feet of the rim, using different releases and spins off the backboard.
        - Scripted combo moves: Choreograph a dribble sequence (behind-the-back into crossover, etc.) and drill it repeatedly until it becomes second nature — Kyrie treats his in-game creativity as rehearsed, not improvised.

        TRAINING QUOTES:
        - 'What I want people to realize is that when I make a move, it's really a simple move.'
        - 'I have counters to every move.'

        KEY SKILLS: Ball handling, ambidextrous control, finishing at the rim, footwork, creative scoring.

        YOUTUBE SEARCH TERMS:
        - 'Kyrie Irving ball handling drills'
        - 'Kyrie Irving finishing moves layup drills'
        - 'Kyrie Irving signature dribble moves'
        """,
        metadata={"player": "Kyrie Irving", "position": "PG", "skills": "ball handling, finishing, creative scoring"}
    ),

    # ─── JAMES HARDEN ─────────────────────────────────────
    Document(
        page_content="""
        Player: James Harden | Team: LA Clippers | Position: Guard

        SIGNATURE DRILLS:
        - Step-back footwork progression: Drill the footwork slowly first — forward dribble, plant, step back, shoot — before adding speed.
        - Step-back off the between-the-legs dribble: Once the basic footwork is clean, add a between-the-legs dribble into the step-back to make the move feel natural in motion.
        - Hesitation into step-back: Use a hesitation dribble to freeze the defender, then explode backward into space for the jumper.
        - Euro step progression: Jab step into a side step (footwork only), then add a dribble, then add full speed and distance from the 3-point line — a layered progression for the Euro step finish.
        - Balance and release drill: Practice the step-back while focused purely on not leaning back too far, keeping the shot's balance and rhythm clean.

        TRAINING QUOTES:
        - 'Are they shaky? Are they moving while I'm dribbling?' (on reading a defender's balance before attacking)

        KEY SKILLS: Step-back shooting, creating space, hesitation moves, footwork, one-on-one scoring.

        YOUTUBE SEARCH TERMS:
        - 'James Harden step back drill tutorial'
        - 'James Harden Euro step footwork drill'
        - 'James Harden workout training routine'
        """,
        metadata={"player": "James Harden", "position": "G", "skills": "step-back, creating space, footwork"}
    ),

    # ─── STEVE NASH ─────────────────────────────────────
    Document(
        page_content="""
        Player: Steve Nash | Team: Phoenix Suns / Dallas Mavericks | Position: Point Guard

        SIGNATURE DRILLS (from Nash's iconic 20-Minute Workout):
        - 50x close to the rim: Start with easy baskets from the lane line under the rim, switching sides with each shot to build a rhythm without stopping.
        - 10x pull-up jumpers: From the top of the key, take a few quick steps toward the basket before a pull-up jumper. Alternate direction with each rebound — pure footwork emphasis.
        - 10x spin and jump shot: From the same starting position, take a few quick steps and add a spin move before each jump shot. Alternate directions with each rebound.
        - 10x college three-point shots: Work around the arc at the NCAA three-point distance (19'9"), keeping the heart rate up between makes.
        - Final series — NBA three-point line: Finish the workout at the regulation NBA three-point distance (23'9"), shooting fatigued to simulate late-game shot-making.
        - The whole routine runs continuously for 20 minutes with no stoppages, building shooting touch under cardio fatigue — exactly the workout in Nash's own training video.

        TRAINING QUOTES:
        - 'Get to it, and keep those eyes up.'

        KEY SKILLS: Shooting under fatigue, footwork into jumpers, conditioning, range progression, rhythm shooting.

        YOUTUBE SEARCH TERMS:
        - 'Steve Nash 20 Minute Workout' (full session: https://www.youtube.com/watch?v=D3cO9c7RgAE)
        - 'Steve Nash shooting drill routine'
        - 'Steve Nash hesitation workout'
        """,
        metadata={"player": "Steve Nash", "position": "PG", "skills": "shooting under fatigue, conditioning, footwork"}
    ),

]

print(f"✅ Knowledge base loaded: {len(player_data)} players ready.")
for doc in player_data:
    print(f"   🏀 {doc.metadata['player']} — {doc.metadata['skills']}")

## Cell 4 — Embed & Store in Vector Database (Chroma)
This is the **A/D conversion + tag database** step.

LangChain converts each player's text into a vector (list of numbers), then Chroma stores them locally on your machine — like a local PLC historian database.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import shutil, os

# ─────────────────────────────────────────────────────────────────
# EMBEDDINGS — converts text to numbers (free, runs locally)
# Think: analog signal → normalized digital value
# ─────────────────────────────────────────────────────────────────
print("⚙️  Loading embedding model (downloads once, ~90MB)...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# ─────────────────────────────────────────────────────────────────
# CHROMA VECTOR DB — local database stored in ./coachbot_db folder
# Think: PLC tag historian database on your local drive
# ─────────────────────────────────────────────────────────────────
DB_PATH = "./coachbot_db"

# Wipe old DB so we start fresh each time you run this cell
if os.path.exists(DB_PATH):
    shutil.rmtree(DB_PATH)

vectordb = Chroma.from_documents(
    documents=player_data,
    embedding=embeddings,
    persist_directory=DB_PATH
)

print(f"✅ Vector DB built! {vectordb._collection.count()} documents stored in {DB_PATH}")
print("   Your player knowledge base is now searchable.")

## Cell 5 — Build the RAG Chain (The Brain)
This is where **LangChain wires everything together** — like rungs in a ladder logic program:

```
User Query → Retriever → Prompt Template → Gemini LLM → Workout Output
```

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ─────────────────────────────────────────────────────────────────
# RETRIEVER — finds the most relevant player doc from the DB
# Think: recipe table lookup — finds the closest match to your query
# ─────────────────────────────────────────────────────────────────
retriever = vectordb.as_retriever(search_kwargs={"k": 2})

# ─────────────────────────────────────────────────────────────────
# GEMINI LLM — the reasoning engine
# Think: the HMI logic layer that reads inputs and generates output (same role, different brand)
# ─────────────────────────────────────────────────────────────────
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0.7
)

# ─────────────────────────────────────────────────────────────────
# PROMPT TEMPLATE — structured instruction to Gemini
# Think: a structured message format, like a PLC alarm message template (unchanged)
# ─────────────────────────────────────────────────────────────────
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are CoachBot, an elite basketball training assistant. You help coaches
design 1-on-1 training sessions for youth players (ages 8-18) inspired by
NBA players' real training methods.

Use ONLY the player information provided below to design the workout.
Include real drills, motivational quotes from the player, and YouTube
search terms so the coach can find videos to show their athlete.

PLAYER KNOWLEDGE BASE:
{context}

COACH'S REQUEST:
{question}

Respond with a complete, structured 60-minute training session in this format:

🏀 COACHBOT SESSION PLAN
Player Inspiration: [Player Name]
Theme: [Core skill focus]

💬 OPENING QUOTE (Read this to your athlete at the start):
[Quote from the player]

📋 SESSION BREAKDOWN:

⏱️ WARMUP (10 min)
[2-3 warmup activities]

🔧 SKILL BLOCK 1 (15 min) — [Skill name]
[Drill name]: [Instructions]
[Coaching cue from the player's style]

🔧 SKILL BLOCK 2 (15 min) — [Skill name]
[Drill name]: [Instructions]
[Coaching cue]

🔧 SKILL BLOCK 3 (10 min) — [Skill name]
[Drill name]: [Instructions]

🏁 COMPETITIVE FINISH (8 min)
[1v1 or scoring challenge tied to session theme]

🧘 COOLDOWN & MINDSET (2 min)
[Closing quote + reflection question for the athlete]

🎥 VIDEOS TO SHOW YOUR ATHLETE:
Search YouTube for:
- [Search term 1]
- [Search term 2]

💡 COACH NOTES:
[2-3 tips for adapting this session to a youth player]
"""
)

# ─────────────────────────────────────────────────────────────────
# RAG CHAIN — wires it all together (LangChain magic)
# Think: the full ladder logic rung — from input coil to output relay (unchanged)
# ─────────────────────────────────────────────────────────────────
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

print("✅ CoachBot RAG chain is ready!")
print("   Run the next cell to generate a workout.")

## Cell 6 — 🏀 Generate a Workout!
Change the `query` below to any player or skill focus you want.

**Example queries:**
- `"Build me a workout inspired by Steph Curry"`
- `"My athlete needs to work on finishing at the rim like LeBron"`
- `"Kobe Bryant inspired mid-range shooting session"`
- `"My 12-year-old needs to work on ball handling — pick the best player for that"`

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 🏀 CHANGE THIS QUERY to generate any workout you want
# ─────────────────────────────────────────────────────────────────
query = "Build me a 60-minute workout for a 13-year-old inspired by Steph Curry. Focus on shooting and ball handling."

# ─────────────────────────────────────────────────────────────────
# RUN THE RAG PIPELINE
# ─────────────────────────────────────────────────────────────────
print("🤖 CoachBot is generating your session plan...\n")
print("=" * 60)

response = rag_chain.invoke(query)
print(response)

print("=" * 60)
print("✅ Session plan complete!")

## Cell 7 — 💾 Save Workout to a Text File (Optional)
Save any generated session plan to a file so you can print it or use it at the gym.

In [ ]:
# Save the last generated workout to a text file
import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
filename = f"session_plan_{timestamp}.txt"

with open(filename, "w") as f:
    f.write(f"Query: {query}\n")
    f.write("=" * 60 + "\n")
    f.write(response)

print(f"✅ Workout saved to: {filename}")

## Cell 8 — ➕ Add a New Player (Template)
Copy this cell and fill in details to add any player to your knowledge base.
Run Cells 4 and 5 again after adding to rebuild the database.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# ADD A NEW PLAYER — copy this block into Cell 3's player_data list
# Then re-run Cell 4 and Cell 5 to rebuild the database
# ─────────────────────────────────────────────────────────────────

new_player_template = Document(
    page_content="""
    Player: [PLAYER NAME] | Team: [TEAM] | Position: [POSITION]

    SIGNATURE DRILLS:
    - [Drill 1 name]: [Instructions]
    - [Drill 2 name]: [Instructions]
    - [Drill 3 name]: [Instructions]

    TRAINING QUOTES:
    - '[Quote 1]'
    - '[Quote 2]'

    KEY SKILLS: [skill1, skill2, skill3]

    YOUTUBE SEARCH TERMS:
    - '[Player name] workout drills'
    - '[Player name] training session'
    """,
    metadata={"player": "[PLAYER NAME]", "position": "[POS]", "skills": "[skills]"}
)

print("📋 Template ready. Copy the Document(...) block above into Cell 3's player_data list.")
print("   Then re-run Cell 4 (embed) and Cell 5 (chain) to update the database.")

---
## 🗺️ What Each LangChain Piece Does (Summary)

| Component | What It Does | Controls Analogy |
|---|---|---|
| `HuggingFaceEmbeddings` | Converts text → numbers | A/D converter |
| `Chroma` | Stores & searches vectors | PLC tag historian DB |
| `retriever` | Finds most relevant player | Recipe/setpoint lookup |
| `PromptTemplate` | Structures the question to Gemini | Structured alarm/message format |
| `ChatGoogleGenerativeAI` | The Gemini AI brain | HMI logic layer |
| `rag_chain` | Wires all pieces together | Full ladder logic rung |
| `StrOutputParser` | Cleans the output | Output relay / formatted display |

---
## 🚀 What to Build Next
1. **Add more players** — fill out the template in Cell 8
2. **Add player weakness input** — e.g. `"My athlete struggles with left hand"` and let CoachBot adapt the drill
3. **Build a Streamlit UI** — a simple web app so you don't need the notebook
4. **Add player notes per athlete** — track progress over sessions
5. **Connect to YouTube API** — auto-fetch real video links instead of search terms